# Model Evaluation and Predictions

In [1]:
# Import libraries
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import random

import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Load Observation Dataset
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'

dataset1 = 'Dataset_RF_Model_SW_COMIDS.csv'
#dataset1 = 'Dataset_RF_Model_GW_COMIDS.csv'

input_data_ = pd.read_csv(future_dir+dataset1)
input_data = input_data_.drop(columns=['COMID','viol_freq']) 
#y = input_data['Viol_Class'].values
#ncols = input_data.shape[1]
#ncols


In [3]:
## Save Preditor Data for SHAP Analysis
X_input_data = input_data.drop(columns=['Viol_Class']).values
X_input_data.shape

#  Convert to tensor data
X_input_data = torch.from_numpy(X_input_data).type(torch.float)
X_input_data.shape,X_input_data.dtype

# Find the size of the smallest class
min_size = input_data['Viol_Class'].value_counts().min()

min_size  = min_size * 10

# Sample exactly 'min_size' elements from each binary group
#balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42).reset_index(drop=True)

# If increasing the min_size, you can use the 'replace=True' argument to allow for sampling with replacement
balanced_df = input_data.groupby('Viol_Class').sample(n=min_size, random_state=42, replace=True).reset_index(drop=True)


# Convert to tensors and split into train and test sets
from sklearn.model_selection import train_test_split
#X = input_data.drop(columns=['Viol_Class']).values # when use this the model just predicts the majority class
#y = input_data['Viol_Class'].values
X = balanced_df.drop(columns=['Viol_Class']).values
y = balanced_df['Viol_Class'].values

# Turn data into tensors
X = torch.from_numpy(X).type(torch.float)
y = torch.from_numpy(y).type(torch.float)

# Make a copy to use later
X_full = X.clone()
y_full = y.clone()

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.2,
                                                    random_state=2
)

nrows = X_train.size()[0]
ncols = X_train.size()[1]

C:\Users\MPennino\AppData\Local\Temp\ipykernel_56108\3652074261.py:30: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at D:\bld\libtorch_1784990676194\work\torch\csrc\utils\tensor_numpy.cpp:219.)
  y = torch.from_numpy(y).type(torch.float)


In [4]:
import torch
import torch.nn as nn

class ImprovedBinaryClassifier(nn.Module):
    def __init__(self, input_dim=ncols, hidden_dim=64):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LeakyReLU(0.1),             # Prevents dead neurons
            nn.BatchNorm1d(hidden_dim),     # Stabilizes training
            
            nn.Linear(hidden_dim, hidden_dim),
            nn.LeakyReLU(0.1),
            nn.Dropout(0.2),                # Prevents overfitting
            
            nn.Linear(hidden_dim, 1)        # Outputs raw logits (No Sigmoid here!)
        )
        
    def forward(self, x):
        return self.network(x)

In [5]:
# Load the Model
future_dir = 'C:/Users/MPennino/OneDrive - Environmental Protection Agency (EPA)/Projects/OASES/Data/Future_NO3/'
model_dir = future_dir + "/Models"
model_dir

filename = '/torch_model_future_NO3_sw.pth'

loaded_model = torch.load(model_dir + filename, weights_only=False)
loaded_model.eval()

ImprovedBinaryClassifier(
  (network): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): LeakyReLU(negative_slope=0.1)
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (3): Linear(in_features=64, out_features=64, bias=True)
    (4): LeakyReLU(negative_slope=0.1)
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [6]:
import torchmetrics

# Define your classification task ('binary', 'multiclass', or 'multilabel')
task = "binary"

# Initialize metrics
sensitivity_metric = torchmetrics.classification.Recall(task=task)
specificity_metric = torchmetrics.classification.Specificity(task=task)

# Get model predictions on the test set
loaded_model.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(loaded_model(X_test))).squeeze()

# get target / observed response values
target = y_test

# Simulated model predictions (logits or probabilities) and ground truth targets
# preds  = torch.tensor([0, 1, 0, 1, 1, 0])
# target = torch.tensor([0, 1, 1, 0, 1, 0])

# Compute metrics
#sensitivity = sensitivity_metric(preds, target)
#specificity = specificity_metric(preds, target)

# Calculate True Positives, True Negatives, False Positives, False Negatives
TP = torch.sum((preds == 1) & (target == 1)).float()
TN = torch.sum((preds == 0) & (target == 0)).float()
FP = torch.sum((preds == 1) & (target == 0)).float()
FN = torch.sum((preds == 0) & (target == 1)).float()

PCC = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN )
specificity = TN / (TN + FP )

print(f"PCC: {PCC.item():.4f}")
print(f"Sensitivity (True Positives): {sensitivity.item():.4f}")
print(f"Specificity (True Negatives): {specificity.item():.4f}")

PCC: 0.9756
Sensitivity (True Positives): 1.0000
Specificity (True Negatives): 0.9494


In [7]:
# AUC Calculation
from sklearn.metrics import roc_auc_score
import numpy as np


loaded_model.eval()
with torch.inference_mode():
    preds = torch.round(torch.sigmoid(loaded_model(X_test))).squeeze()

# get target / observed response values
target = y_test

preds2 = preds.detach().cpu().tolist()
target2 = target.detach().cpu().tolist()

len(preds), len(target), type(preds), type(preds2), target2[0:5], preds2[0:5]

# 2. Calculate the AUC Score
auc_score = roc_auc_score(target2, preds2)
print(f"Test AUC: {auc_score:.4f}")

Test AUC: 0.9747


# Make Predictions

In [8]:
# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()
names_list

['PopDen2010Ws',
 'PctForest2019Ws',
 'PctCrop2019Ws',
 'precip9120ws',
 'tmean9120ws',
 'BFIWs',
 'permws',
 'N_TW2012Ws',
 'RockNWs',
 'N_Surp_kgsqkm_2017ws',
 'ElevWs',
 'Fe2O3Ws',
 'NHDslope_Pct_Ws']

# Scenario: Current Period (base year 2020)

In [9]:
# Load Prediction Dataset
dataset = "current_NO3_predictors_catchments.parquet"
pred_data_ = pd.read_parquet(future_dir+dataset) # Read a single Parquet file
#pred_data_ = pd.read_table(future_dir+dataset) # Read a single Parquet file

# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()
#print(names_list)

# Remove extra fields (model predictions won't work unless the prediction dataset has the same fields as the training dataset)
pred_data = pred_data_[names_list]
#pred_data.head(3)
len(pred_data), pred_data['N_Surp_kgsqkm_2017ws'].mean()

(2643994, np.float64(2579.6611209442385))

In [10]:
# Convert to Tensor
PRED_DATA = pred_data.values

# Turn data into tensors
PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

# Make Predictions on full HUC12 dataset, for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs = torch.sigmoid(loaded_model(PRED_DATA)).squeeze()
    
# Convert from torch to pandas dataframe
y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with HUC12 
result2 = pd.concat([pred_data_['COMID'], y_probs_df], axis=1)
#len(result2), result2['Pred_Viol_Prob'].mean(), 100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2)
print(f"Mean Prediction Probability: {result2['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2):.2f}%")

Mean Prediction Probability: 0.1578, Violation Rate: 15.44%


In [189]:
# Save Dataset
filename = 'torch_predictions_current_All_COMID_sw.parquet'
#filename = 'torch_predictions_current_All_COMID_gw.parquet'

table = pa.Table.from_pandas(result2)
pq.write_table(table, future_dir + filename)

# Scenario: 50% increase in N Surplus

In [190]:
pred_data_nsurp = pred_data.copy()
pred_data_nsurp['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp['N_Surp_kgsqkm_2017ws'] * 1.5
pred_data_nsurp['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()

(np.float64(3869.491681416358), np.float64(2579.6611209442385))

In [191]:

# Convert to Tensor
PRED_DATA = pred_data_nsurp.values

# Turn data into tensors
PRED_DATA = torch.from_numpy(PRED_DATA).type(torch.float)

# Make Predictions on full HUC12 dataset, for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs = torch.sigmoid(loaded_model(PRED_DATA)).squeeze()
    
# Convert from torch to pandas dataframe
y_probs_df = pd.DataFrame(y_probs.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with HUC12 
result2 = pd.concat([pred_data_['COMID'], y_probs_df], axis=1)
#result2['Pred_Viol_Prob'].mean(), 100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2)
print(f"Mean Prediction Probability: {result2['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result2['Pred_Viol_Prob'] > 0.5).sum() / len(result2):.2f}%")

Mean Prediction Probability: 0.2360, Violation Rate: 23.18%


In [192]:
# Save Dataset
filename = 'torch_predictions_150perc_NSurp_All_COMID_sw.parquet'
#filename = 'torch_predictions_current_All_COMID_gw.parquet'

table = pa.Table.from_pandas(result2)
pq.write_table(table, future_dir + filename)

# Scenario: 50% Reduction in N Surplus

In [193]:
pred_data_nsurp = pred_data.copy()
pred_data_nsurp['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp['N_Surp_kgsqkm_2017ws'] * 0.5
pred_data_nsurp['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()


(np.float64(1289.8305604721193), np.float64(2579.6611209442385))

In [194]:
# Convert to Tensor
PRED_DATA_SURP = pred_data_nsurp.values

# Turn data into tensors
PRED_DATA_SURP = torch.from_numpy(PRED_DATA_SURP).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_surp = torch.sigmoid(loaded_model(PRED_DATA_SURP)).squeeze()

# Convert to pd dataframe
y_probs_surp_df = pd.DataFrame(y_probs_surp.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID 
result_surp = pd.concat([pred_data_['COMID'], y_probs_surp_df], axis=1)
print(f"Mean Prediction Probability: {result_surp['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_surp['Pred_Viol_Prob'] > 0.5).sum() / len(result_surp):.2f}%")

Mean Prediction Probability: 0.1538, Violation Rate: 15.07%


In [195]:
# Save Results
filename = 'torch_pred_scenario_50perc_NSurp_COMID_sw.parquet'
#filename = 'torch_pred_scenario_nue_COMID_gw.parquet'

table = pa.Table.from_pandas(result_surp)
pq.write_table(table, future_dir + filename)

# Scenario: Complete removal of N Surplus


In [196]:
pred_data_nsurp = pred_data.copy()
pred_data_nsurp['N_Surp_kgsqkm_2017ws'] = pred_data_nsurp['N_Surp_kgsqkm_2017ws'] * 0
pred_data_nsurp['N_Surp_kgsqkm_2017ws'].mean(), pred_data['N_Surp_kgsqkm_2017ws'].mean()

(np.float64(0.0), np.float64(2579.6611209442385))

In [197]:
# Convert to Tensor
PRED_DATA_SURP = pred_data_nsurp.values

# Turn data into tensors
PRED_DATA_SURP = torch.from_numpy(PRED_DATA_SURP).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_surp = torch.sigmoid(loaded_model(PRED_DATA_SURP)).squeeze()

# Convert to pd dataframe
y_probs_surp_df = pd.DataFrame(y_probs_surp.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID 
result_surp = pd.concat([pred_data_['COMID'], y_probs_surp_df], axis=1)
print(f"Mean Prediction Probability: {result_surp['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_surp['Pred_Viol_Prob'] > 0.5).sum() / len(result_surp):.2f}%")

Mean Prediction Probability: 0.0792, Violation Rate: 7.79%


In [198]:
# Save Results
filename = 'torch_pred_scenario_0perc_NSurp_COMID_sw.parquet'
#filename = 'torch_pred_scenario_nue_COMID_gw.parquet'

table = pa.Table.from_pandas(result_surp)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 15%

In [202]:
filename = 'Scenario_NUE15_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

((2644024, 13), np.float64(1984.880753218651))

In [203]:
# Convert to Tensor
PRED_DATA_NUE = pred_data_nue.values

# Turn data into tensors
PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# Convert to pd dataframe
y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID 
result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
print(f"Mean Prediction Probability: {result_nue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue):.2f}%")

Mean Prediction Probability: 0.1618, Violation Rate: 15.83%


In [204]:
# Save Results
filename = 'torch_pred_scenario_nue15_COMID_sw.parquet'
#filename = 'torch_pred_scenario_nue15_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 25%

In [209]:
filename = 'Scenario_NUE25_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

((2644024, 13), np.float64(1616.641929838698))

In [210]:
# Convert to Tensor
PRED_DATA_NUE = pred_data_nue.values

# Turn data into tensors
PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# Convert to pd dataframe
y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID 
result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
print(f"Mean Prediction Probability: {result_nue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue):.2f}%")

Mean Prediction Probability: 0.1926, Violation Rate: 18.94%


In [ ]:
# Save Results
filename = 'torch_pred_scenario_nue25_COMID_sw.parquet'
#filename = 'torch_pred_scenario_nue_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased NUE by 50%

In [214]:
filename = 'Scenario_NUE50_Dataset_Cat.parquet'
pred_data_nue_ = pq.read_table(future_dir + filename)
pred_data_nue1 = pred_data_nue_.to_pandas()
#pred_data_nue1.columns
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['COMID']) 
pred_data_nue = pred_data_nue1[names_list]
pred_data_nue.shape, pred_data_nue['N_Surp_kgsqkm_2017ws'].mean()

((2644024, 13), np.float64(696.0448713888147))

In [215]:
# Convert to Tensor
PRED_DATA_NUE = pred_data_nue.values

# Turn data into tensors
PRED_DATA_NUE = torch.from_numpy(PRED_DATA_NUE).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_nue = torch.sigmoid(loaded_model(PRED_DATA_NUE)).squeeze()

# Convert to pd dataframe
y_probs_nue_df = pd.DataFrame(y_probs_nue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID 
result_nue = pd.concat([pred_data_nue1['COMID'], y_probs_nue_df], axis=1)
#result_nue['Pred_Viol_Prob'].mean(), 100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue)
print(f"Mean Prediction Probability: {result_nue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_nue['Pred_Viol_Prob'] > 0.5).sum() / len(result_nue):.2f}%")

Mean Prediction Probability: 0.1308, Violation Rate: 12.77%


In [ ]:
# Save Results
filename = 'torch_pred_scenario_nue_COMID_sw.parquet'
#filename = 'torch_pred_scenario_nue_COMID_gw.parquet'

table = pa.Table.from_pandas(result_nue)
pq.write_table(table, future_dir + filename)

# Scenario: Increased Productivity (55% increase, constant NUE)

In [11]:
#pred_data_prod = pred_data.copy()
#pred_data_prod['n_surplus_kgsqkm'] = pred_data_prod['n_surplus_kgsqkm'] * 2
#pred_data_prod['n_surplus_kgsqkm'].mean(), pred_data['n_surplus_kgsqkm'].mean()

filename = 'Scenario_Prod55_NUE0_Dataset_Cat.parquet'
pred_data_prod_ = pq.read_table(future_dir + filename)
pred_data_prod1 = pred_data_prod_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_prod = pred_data_prod1[names_list]
pred_data_prod.shape

(2644024, 13)

In [14]:
# Convert to Tensor
PRED_DATA_PROD = pred_data_prod.values

# Turn data into tensors
PRED_DATA_PROD = torch.from_numpy(PRED_DATA_PROD).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_prod = torch.sigmoid(loaded_model(PRED_DATA_PROD)).squeeze()

# Convert to pd dataframe
y_probs_prod_df = pd.DataFrame(y_probs_prod.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with HCOMID
result_prod = pd.concat([pred_data_prod1['COMID'], y_probs_prod_df], axis=1)
print(f"Mean Prediction Probability: {result_prod['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_prod['Pred_Viol_Prob'] > 0.5).sum() / len(result_prod):.2f}%")

Mean Prediction Probability: 0.2092, Violation Rate: 20.54%


In [ ]:
# Save Results
filename = 'torch_pred_scenario_prod55_nue0_HUC12_sw.parquet'
#filename = 'torch_pred_scenario_prod_HUC12_gw.parquet'

table = pa.Table.from_pandas(result_prod)
pq.write_table(table, future_dir + filename)

# Scenario: Increased Productivity & NUE (55% increase, 15 Increase NUE)

In [20]:
filename = 'Scenario_Prod55_NUE15_Dataset_Cat.parquet'
pred_data_prod_ = pq.read_table(future_dir + filename)
pred_data_prodnue1 = pred_data_prod_.to_pandas()
#pred_data_nue1.head(3)
#pred_data_nue = pred_data_nue1.drop(columns=['HUC12']) 
pred_data_prodnue = pred_data_prodnue1[names_list]
pred_data_prodnue.shape

(2644024, 13)

In [21]:
# Convert to Tensor
PRED_DATA_PRODNUE = pred_data_prodnue.values

# Turn data into tensors
PRED_DATA_PRODNUE = torch.from_numpy(PRED_DATA_PRODNUE).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_prodnue = torch.sigmoid(loaded_model(PRED_DATA_PRODNUE)).squeeze()

# Convert to pd dataframe
y_probs_prodnue_df = pd.DataFrame(y_probs_prodnue.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with COMID
result_prodnue = pd.concat([pred_data_prodnue1['COMID'], y_probs_prodnue_df], axis=1)
print(f"Mean Prediction Probability: {result_prodnue['Pred_Viol_Prob'].mean():.4f}, Violation Rate: {100 * (result_prodnue['Pred_Viol_Prob'] > 0.5).sum() / len(result_prodnue):.2f}%")

Mean Prediction Probability: 0.1507, Violation Rate: 14.71%


In [22]:
# Save Results
filename = 'torch_pred_scenario_prod55_nue15_HUC12_sw.parquet'
#filename = 'torch_pred_scenario_prod_HUC12_gw.parquet'

table = pa.Table.from_pandas(result_prodnue)
pq.write_table(table, future_dir + filename)

# Scenario: Future projection

In [ ]:
# Load Dataset
#filename = 'Dataset_Future_Catchment_4.5GISS.parquet'
#filename = 'Dataset_Future_Catchment_8.5GISS.parquet'
#filename = 'Dataset_Future_Catchment_4.5Hadgem.parquet'
filename = 'Dataset_Future_Catchment_8.5Hadgem.parquet'

# Get list of predictors from training dataset
temp = input_data.drop(columns=['Viol_Class'])
names_list = temp.columns.tolist()

pred_data_fut_ = pq.read_table(future_dir + filename)
pred_data_fut1 = pred_data_fut_.to_pandas()
pred_data_fut1.shape
pred_data_fut1.columns
pred_data_fut = pred_data_fut1[names_list]
pred_data_fut.shape

In [102]:
# Convert to Tensor
PRED_DATA_FUT = pred_data_fut.values

# Turn data into tensors
PRED_DATA_FUT = torch.from_numpy(PRED_DATA_FUT).type(torch.float)

# Make Predictions for prediction probabilities
loaded_model.eval()
with torch.inference_mode():
    y_probs_fut = torch.sigmoid(loaded_model(PRED_DATA_FUT)).squeeze()

# Convert to pd dataframe
y_probs_fut_df = pd.DataFrame(y_probs_fut.cpu().numpy(), columns=['Pred_Viol_Prob'])

# Merge with Catchment 
result_fut = pd.concat([pred_data_fut1['COMID'], y_probs_fut_df], axis=1)
result_fut.shape, result_fut['Pred_Viol_Prob'].mean(), 100 * (result_fut['Pred_Viol_Prob'] > 0.5).sum() / len(result_fut)

((2644024, 2), np.float32(0.14001696), np.float64(13.078776894612151))

In [89]:
# Save Results
#filename = 'torch_pred_scenario_fut_RCP4.5G_COMID_sw.parquet'
#filename = 'torch_pred_scenario_fut_RCP8.5G_COMID_sw.parquet'
#filename = 'torch_pred_scenario_fut_RCP4.5H_COMID_sw.parquet'
filename = 'torch_pred_scenario_fut_RCP8.5H_COMID_sw.parquet'


table = pa.Table.from_pandas(result_fut)
pq.write_table(table, future_dir + filename)

# Scenario: Future Projection & Increased Prod & NUE